# 05. Testes de Carga e Validação de Latência

**Objetivo:** Comprovar que o modelo de arquitetura Medallion construído mitiga a latência de leitura analítica. Vamos comparar a execução de uma consulta de Grafos (encontrar coautores) rodando diretamente na camada Bronze (JSON aninhado) contra a camada Ouro (Grafos colunares em Delta).

In [0]:
import time
from pyspark.sql.functions import col, explode

print("TESTE DE PERFORMANCE: RECUPERAÇÃO DE COAUTORIAS")
print("Objetivo: Encontrar todos os autores que colaboraram em artigos com um autor específico.")

# Vamos escolher um autor famoso como cobaia (Você pode trocar por outro se quiser)
TARGET_AUTHOR = "https://openalex.org/A5084515381"
print(f"Autor Alvo Escolhido: {TARGET_AUTHOR}")

# IMPORTANTE: Usamos 'spark.catalog.clearCache()' para garantir que o Spark 
# não trapaceie guardando os dados na memória RAM entre os testes!
#spark.catalog.clearCache()

In [0]:
# TESTE 1: CONSULTA LENTA (RAW JSON / CAMADA BRONZE)

print("\n[1] Iniciando consulta na Camada Bronze (JSON Aninhado)...")
inicio_bronze = time.time()

df_bronze = spark.table("bronze_openalex")

# Precisamos usar a função 'exists' dentro do array estruturado para achar os artigos dele
df_bronze_filtered = df_bronze.filter(
    f"exists(authorships, a -> a.author.id == '{TARGET_AUTHOR}')"
)

# Depois precisamos 'explodir' o array de autores dos artigos encontrados
df_bronze_coautores = df_bronze_filtered.select(explode("authorships").alias("auth")) \
    .select(col("auth.author.id").alias("coautor_id"), col("auth.author.display_name").alias("nome")) \
    .filter(col("coautor_id") != TARGET_AUTHOR) \
    .dropDuplicates(["coautor_id"])

# Executa a ação fisicamente para contar o tempo
count_bronze = df_bronze_coautores.count()
fim_bronze = time.time()
tempo_bronze = fim_bronze - inicio_bronze

print(f"Resultado Bronze: Encontrados {count_bronze} coautores.")
print(f"Tempo de Execução Bronze: {tempo_bronze:.4f} segundos")

In [0]:
# TESTE 2: CONSULTA OTIMIZADA (ARQUITETURA MEDALLION / CAMADA OURO)

print("\n[2] Iniciando consulta na Camada Ouro (Delta/Grafos)...")
inicio_ouro = time.time()

df_arestas = spark.table("gold_arestas")
df_vertices = spark.table("gold_vertices")

# Lógica de Grafo Relacional:
# 1. Pega os IDs dos artigos que o Alvo escreveu
df_artigos_alvo = df_arestas.filter(
    (col("tipo_aresta") == "Autoria") & (col("id_origem") == TARGET_AUTHOR)
).select(col("id_destino").alias("work_id"))

# 2. Cruza (JOIN) para achar quem mais tem Autoria nesses mesmos artigos
df_coautorias = df_arestas.filter(col("tipo_aresta") == "Autoria") \
    .join(df_artigos_alvo, df_arestas.id_destino == df_artigos_alvo.work_id, "inner") \
    .filter(col("id_origem") != TARGET_AUTHOR) \
    .select(col("id_origem").alias("coautor_id")) \
    .dropDuplicates(["coautor_id"])

# 3. Cruza com os Vértices apenas para pegar o Nome do autor
df_ouro_coautores = df_coautorias.join(df_vertices, df_coautorias.coautor_id == df_vertices.id_no, "inner") \
    .select(col("id_no"), col("nome_no"))

count_ouro = df_ouro_coautores.count()
fim_ouro = time.time()
tempo_ouro = fim_ouro - inicio_ouro

print(f"Resultado Ouro: Encontrados {count_ouro} coautores.")
print(f"Tempo de Execução Ouro: {tempo_ouro:.4f} segundos")